**2026-10-09 update:** This collection contains repeated `vid_<hash>_face_...` filename families. The original per-image manifests permit family overlap. Follow `RGB_GENERALIZATION_REPAIR.md` before another training run; keep existing hashes/backups. The repaired split does not by itself remove source/format bias.

# Colab session 1 — CPU preparation

Run this notebook with **CPU**; no GPU is used for hashing or selection. Use a high-RAM runtime if available: the current audit/recovery tools hold large metadata tables in RAM. Runtime/disk availability varies; monitor RAM while preparing a million images.

Push/upload the new repository code before cloning. These cells were authored and syntax-checked, NOT executed here.

Input is exactly `/content/dataset/Preapred Dataset/train`. This workflow intentionally uses ONLY that folder, reserves a balanced 15K development set from it, hashes the entire supplied pool once, quarantines label conflicts and removes train/dev overlap, then derives nested 100K, 500K, 1M-target and `all` training manifests. Every size shares the same cleaned dev. No images are copied by manifest preparation. Sizes report actual counts: 1M total input cannot yield 1M train after reserving dev. Target-sized manifests are balanced where available; `all` includes all eligible training rows and may be imbalanced.

Groups are explicitly image-level and unverified for people/videos. Hashes detect exact duplicates, not all near-duplicates. This remains an engineering experiment, not verified source-held-out evidence. Reserve untouched external data for final generalization evaluation. Do not add an external test to this pool.

Keep dataset bytes/relative names unchanged throughout preparation and future sessions. If you re-extract after interruption, mtime changes may require rehashing for an INCOMPLETE audit; a completed saved preparation does not need hashing again under the unchanged-dataset assumption.

Google Drive saves live under `MyDrive/deepfake_rgb`. Images must be restored to local `/content` each session; manifests do not contain images. Avoid training from hundreds of thousands of small Drive files; copy/extract an archive locally ([Colab guidance](https://research.google.com/colaboratory/faq.html)).


## 1. Mount Drive and set paths

Run at the start of the CPU session.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, shutil, subprocess, sys, hashlib, zipfile
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/deepfake_rgb')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path('/content/Novel-Deepfake-Detection')
DATA_ROOT = Path('/content/dataset/Preapred Dataset/train')
PREP_ROOT = Path('/content/prepared_data/rgb_sizes_seed42')
os.environ.update(REPO_ROOT=str(REPO_ROOT), DATA_ROOT=str(DATA_ROOT), PREP_ROOT=str(PREP_ROOT))
print('Persistent storage:', DRIVE_ROOT)
print('Required images:', DATA_ROOT)


## 2. Get updated repository

Latest Colab helper must be on the branch, or upload the updated repository at REPO_ROOT.


In [ ]:
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', 'GeneralizationDeepFakeDetection',
                    'https://github.com/AbdelrahmanHassan131/Novel-Deepfake-Detection.git', str(REPO_ROOT)], check=True)
assert (REPO_ROOT / 'tools/colab_manifest_sizes.py').is_file(), 'Push/upload the latest Colab changes first.'
subprocess.run(['git', '-C', str(REPO_ROOT), 'log', '-1', '--oneline'], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 3. Dependencies

Keep Colab’s installed torch/torchvision. CPU preparation does not require CUDA.


In [ ]:
%pip install -q numpy pillow pytorch-wavelets PyWavelets tensorboard pytest


## 4. Metadata helper checks

Run these small metadata-only tests before scanning images. No GPU or image dataset is used by these tests.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
python -m pytest -q tests/test_colab_manifest_sizes.py tests/test_prepared_manifest_recovery.py


## 5. Provide the images

Download/extract your original dataset before this check. The download command depends on where you store the dataset; do not rename class folders or recompress images. No old checkpoint or manifest is needed.


In [ ]:
# Put/download/extract your original dataset here before running this cell.
# Required: /content/dataset/Preapred Dataset/train/real and .../fake
# Also supported by inventory: 0_real and 1_fake instead of real and fake.
assert DATA_ROOT.is_dir(), f'Extract the image dataset first: {DATA_ROOT}'
class_dirs = [p.name for p in DATA_ROOT.iterdir() if p.is_dir()]
assert ({'real', 'fake'} <= set(class_dirs) or {'0_real', '1_fake'} <= set(class_dirs)), class_dirs
print('Class folders:', class_dirs)
print('Free local disk (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))
print('CPU count:', os.cpu_count())
print('RAM:', next(line.strip() for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:')))


## 6. Restore interrupted work and define backups

Safe only for this same dataset and seed. Completed-stage backups are separate from periodic hash-cache snapshots.


In [ ]:
# Optional recovery after an interrupted CPU session; keep the same images/settings.
PREP_ROOT.mkdir(parents=True, exist_ok=True)
backup = DRIVE_ROOT / 'preparation_progress.zip'
if backup.exists() and not any(PREP_ROOT.iterdir()):
    with zipfile.ZipFile(backup) as archive:
        archive.extractall(PREP_ROOT)
    print('Restored completed preparation stages.')
cache_backup = DRIVE_ROOT / 'hashes_progress.sqlite'
if cache_backup.exists() and not (PREP_ROOT / 'hashes.sqlite').exists():
    shutil.copy2(cache_backup, PREP_ROOT / 'hashes.sqlite')
    print('Restored hash cache. Changed file sizes/mtimes trigger safe rehashing.')

def save_preparation(name='preparation_progress.zip', include_cache=False):
    local = Path('/content') / name
    with zipfile.ZipFile(local, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for file in PREP_ROOT.rglob('*'):
            if file.is_file() and (include_cache or not file.name.startswith('hashes.sqlite')):
                archive.write(file, file.relative_to(PREP_ROOT))
    destination = DRIVE_ROOT / name
    temporary = destination.with_suffix('.uploading')
    shutil.copy2(local, temporary)
    temporary.replace(destination)
    print('Saved to Drive:', destination)


## 7. Inventory and reserve development data

Scans all directory entries but does not hash images yet. Class mapping: real=0, fake=1. Provisional per-image split is cleaned after the full audit.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
if [[ ! -f "$PREP_ROOT/pool_manifest.csv" ]]; then
  python -u prepare_dataset.py inventory --root "$DATA_ROOT" --source diffgan \
    --output "$PREP_ROOT/pool_manifest.csv.tmp" --image_level_groups
  mv "$PREP_ROOT/pool_manifest.csv.tmp" "$PREP_ROOT/pool_manifest.csv"
fi
if [[ ! -f "$PREP_ROOT/candidate.csv" ]]; then
  python -u tools/colab_manifest_sizes.py partition \
    --inventory "$PREP_ROOT/pool_manifest.csv" --output "$PREP_ROOT/candidate.csv" \
    --dev_size 15000 --seed 42
fi


## 8. Save inventory before the long audit

Prevents losing inventory/partition selection if the CPU runtime disconnects.


In [ ]:
save_preparation()


## 9. Hash the entire pool ONCE

Main CPU/disk cost. Progress every 5,000 images; transaction-consistent hash-cache snapshots go to Drive every five minutes. Audit exit 1 is not ignored: recovery below accepts only supported duplicate/label-conflict cases. Other failures stop.


In [ ]:
import threading, sqlite3
from uuid import uuid4

# SQLite stays on local disk. Make transaction-consistent snapshots for Drive.
stop_backup = threading.Event()
def snapshot_hashes():
    cache = PREP_ROOT / 'hashes.sqlite'
    if not cache.exists():
        return
    local = Path('/content/hashes_snapshot.sqlite')
    with sqlite3.connect(str(cache), timeout=60) as source:
        with sqlite3.connect(str(local)) as destination:
            source.backup(destination, pages=1024)
    remote = DRIVE_ROOT / 'hashes_progress.sqlite'
    temporary = remote.with_suffix('.uploading')
    shutil.copy2(local, temporary)
    temporary.replace(remote)
    print('Hash progress saved to Drive.', flush=True)

def backup_worker():
    while not stop_backup.wait(300):
        try:
            snapshot_hashes()
        except Exception as exc:
            print('Drive snapshot failed; local hashing continues:', exc, flush=True)

def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

candidate = PREP_ROOT / 'audited_candidate.csv'
report_path = PREP_ROOT / 'full_audit.json'
completed = False
if report_path.exists() and candidate.exists():
    evidence = json.loads(report_path.read_text())
    completed = evidence.get('hashes_verified') is True and evidence.get('manifest_sha256') == file_digest(candidate)
if completed:
    print('Reusing completed full-pool hash audit; no images reopened.')
else:
    shutil.copy2(PREP_ROOT / 'candidate.csv', candidate)
    worker = threading.Thread(target=backup_worker, daemon=True)
    worker.start()
    fresh_report = PREP_ROOT / ('audit_' + uuid4().hex[:8] + '.json')
    try:
        result = subprocess.run([sys.executable, '-u', 'prepare_dataset.py', 'audit',
            '--manifest', str(candidate), '--root', str(DATA_ROOT), '--output', str(fresh_report),
            '--hashes', '--hash_cache', str(PREP_ROOT / 'hashes.sqlite')], cwd=REPO_ROOT)
    finally:
        stop_backup.set()
        worker.join()
        snapshot_hashes()
    if result.returncode not in (0, 1) or not fresh_report.is_file():
        raise RuntimeError('Audit interrupted/failed. Retain cache and retry; do not proceed.')
    evidence = json.loads(fresh_report.read_text())
    assert evidence.get('hashes_verified') is True
    assert evidence.get('manifest_sha256') == file_digest(candidate), 'Incomplete/mismatched evidence'
    shutil.copy2(fresh_report, report_path)
save_preparation()
print('Completed audit saved. Next: quarantine and derive all training sizes.')


## 10. Clean and derive every size

No images opened. Contradictory components are excluded, never relabeled. Whole duplicate groups remain together; the same dev membership is used across all sizes. A shortfall is reported, never padded with repeated images.


In [ ]:
%%bash
set -euo pipefail
cd "$REPO_ROOT"
python -u tools/recover_prepared_manifest.py \
  --manifest "$PREP_ROOT/audited_candidate.csv" --audit_report "$PREP_ROOT/full_audit.json" \
  --output_dir "$PREP_ROOT/clean_parent" \
  --immutable_dataset --allow_shortfall --quarantine_label_conflicts
python -u tools/colab_manifest_sizes.py subsets \
  --parent "$PREP_ROOT/clean_parent/selected_manifest.csv" --output_dir "$PREP_ROOT/sizes" \
  --sizes 100000 500000 1000000 all --seed 42 --immutable_dataset


## 11. Save final preparation and exact code to Drive

Wait for READY ON DRIVE for both ZIPs before ending the CPU session. Save the image archive separately; the preparation archive is metadata only.


In [ ]:
summary = json.loads((PREP_ROOT / 'sizes/sizes_summary.json').read_text())
for size, row in summary.items():
    print(size, '-> actual train:', row['actual_training_samples'], 'dev:', row['dev_samples'], 'shortfall:', row['shortfall'])
save_preparation('prepared_rgb_sizes_seed42.zip', include_cache=True)

# Preserve the exact code used, so session 2 does not accidentally use a newer checkout.
files = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'ls-files', '-co', '--exclude-standard', '-z']).decode().split('\0')
code_zip = Path('/content/prepared_rgb_code.zip')
with zipfile.ZipFile(code_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for relative in sorted(set(files)):
        file = REPO_ROOT / relative
        if file.is_file() and file.suffix.lower() in {'.py', '.toml', '.yaml', '.yml', '.json', '.md', '.txt', '.ipynb'} and not any(part in {'output', 'checkpoints', '.venv', '.git', 'logs'} for part in Path(relative).parts):
            archive.write(file, relative)
shutil.copy2(code_zip, DRIVE_ROOT / code_zip.name)
for name in ('prepared_rgb_sizes_seed42.zip', 'prepared_rgb_code.zip'):
    path = DRIVE_ROOT / name
    assert path.is_file() and path.stat().st_size > 0
    print('READY ON DRIVE:', path, 'bytes:', path.stat().st_size)
print('CPU session finished. Keep your original image dataset/archive too.')
